In [0]:
from pyspark.sql.functions import *

In [0]:
silver_streaming_df = spark.readStream.format("delta") \
    .table("airspace_pulse.silver.flight_states_enriched")


In [0]:
anomaly_df = silver_streaming_df.filter(
    col("squawk").isin("7500", "7600", "7700") | 
    ((col("on_ground") == True) & (col("vertical_rate") != 0))
    ) \
                .select(
                    col("icao24"),
                    col("callsign"),
                    col("time").alias("event_time"),
                    when(col("squawk") == "7500", lit("Hijack"))
                        .when(col("squawk") == "7600", lit("Radio Failure"))
                        .when(col("squawk") == "7700", lit("Emergency"))
                        .otherwise(lit("Grounded"))
                        .alias("anomaly_type"),
                    when(col("squawk").isin("7700", "7500"), lit("critical"))
                        .otherwise(lit("warning"))
                        .alias("severity"),
                    concat(lit("Squawk "), col("squawk")).alias("detail"),
                    col("longitude"),
                    col("latitude"),
                    current_timestamp().alias("detected_at")
                )


In [0]:
query = anomaly_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", "s3://airspace-lakehouse/airspace_pulse/_checkpoints/silver_flight_states_annomalies") \
    .toTable("airspace_pulse.silver.flight_states_anomalies")

query.awaitTermination()